# Week 9 Assignment — An Emergency Department

## The task

Build a **discrete event simulation of an emergency department**. Patients
arrive at random times with a severity of 1, 2 or 3 chosen at random (1 is the most life-threatening). A limited number
of doctors see them, most severe first and, within a severity, in the order
they arrived. The waiting room holds a limited number of patients. You will
measure how long patients of each severity wait, how full the waiting room
gets, and how wait time and waiting room population change as you change the number of doctors.

## The waiting room is given to you

The waiting room is a **minimum heap**, a priority queue stored in a list.
**You do not build it.** It is given below, complete and explained line by
line, and it is already tested. Your job is to *use* it: instantiate one, `add`
patients to it, and `get` the next patient from it. Read the explanation so
you know what it promises, and read the code if you are curious how it keeps
those promises. The notes in `heaps.ipynb` cover the ideas behind it.

## How this notebook works

- This notebook is the **handout**: the explanation, the given heap, the
  scaffold and the tests. You do not turn it in. You write your solution in Python files
  (see *What to submit* at the end), starting by copying the cells below into
  them. Use the notebook, or any editor, to try things out.
- Code cells marked **Given** are complete. Copy them into your files
  unchanged (the heap, the tests and the report functions).
- Code cells marked **TODO** are yours to complete.
- The tests are near the end, in a section called *Testing*. They are plain
  `assert` functions. Each test prints `ok` or `FAIL` with a reason. Re-run
  the test cell after every change. The simulation tests fail on the
  untouched scaffold; that is expected.
- Run the notebook top to bottom. If you change a cell above, re-run
  everything below it.

As always, every function and method has **exactly one `return`**: set a
result variable to the "didn't work" value first, and overwrite it only once
the guard condition passes.

---

## The waiting room: `MinHeap363`

*This whole section is given. You do not write any of it.*

### What it does

A priority queue is a line where the most important value gets served first,
whatever order the values arrived in. Ours always serves the **smallest**
value first. That is why a patient with severity `1` comes out before a
patient with severity `3`.

| Method | Success | Failure (nothing changes) |
|---|---|---|
| `add(value)` | places `value` in the heap | — (`add` cannot fail) |
| `get()` | removes and returns the smallest value | heap is empty → returns `None` |
| `peek()` | returns the smallest value **without** removing it | heap is empty → returns `None` |
| `size()` | returns the number of values in the heap | — |

### How it is stored

Picture a pyramid of boxes with the smallest number on top. Every box is
smaller than, or equal to, the two boxes right under it. That rule is the
**heap property**. We keep the pyramid in an ordinary Python list, level by
level, left to right, the root in `storage[0]`:

```python
storage = [1, 2, 4, 8]       # 1 on top; 2 and 4 under it; 8 under the 2
```

The parent and children of a position come from arithmetic, not pointers:

\begin{align}
\text{left}(p) & = 2p+1 \\
\text{right}(p) & = 2p+2 \\
\text{parent}(c) & = \left\lfloor \frac{c-1}{2} \right\rfloor
\end{align}

```python
(2 * 0 + 1, 2 * 0 + 2)       # children of position 0 (the 1): positions 1 and 2
(3 - 1) // 2                 # parent of position 3 (the 8): position 1 (the 2)
```

### How it keeps the rule

Adding and removing can break the heap property in exactly one place each, and
each has a repair that follows one value along one path.

- **`add` and `_bubble_up`.** `add` puts the new value at the end of the list,
  the next free spot at the bottom of the pyramid. If it is smaller than its
  parent, it is in the wrong place, so we swap them, and repeat from the new
  spot. It floats up like a cork:

  ```python
  while position > 0 and storage[position] < storage[parent]:
      swap them, then continue from the parent's position
  ```

- **`get` and `_sink_down`.** The answer is the root. We take it out, then move
  the **last** value of the list to the root, so the list stays packed with
  no gaps. That value is probably too big for the top. We compare it with its
  **smaller** child; if it is greater than that child we swap, and repeat from
  the child's position. It sinks like a stone until it is smaller than its
  children or reaches a leaf (a position with no children):

  ```python
  while the value is not at a leaf:
      pick the smaller child
      if the value is greater than that child: swap, then continue from the child
      otherwise: stop
  ```

Each repair moves along one root-to-leaf path, so each costs at most the height
of the tree, about $\log_2 n$ steps.

### Values that are pairs

The heap compares values only with `<` and `>`, so it works for anything Python
can compare, including pairs like `(2, 14)`. Python compares pairs element by
element: the first elements decide, and only on a tie does the second element
count. The simulation uses this below.

In [ ]:
# Given -- a complete minimum heap. Copy it into heap363.py unchanged.
class MinHeap363:
    """A priority queue that always returns its smallest value, stored as a
    complete binary tree in a Python list.

    storage -- the tree, level by level, left to right; the root is storage[0]
    """

    def __init__(self):
        # The list starts empty: a new heap holds nothing.
        self.storage: list = []

    # ---- index arithmetic and swapping

    def _left_child(self, parent: int) -> int:
        """Return the position of the left child of the node at `parent`."""
        # Each level doubles, so the children of position p sit at 2p+1 and 2p+2.
        return 2 * parent + 1

    def _right_child(self, parent: int) -> int:
        """Return the position of the right child of the node at `parent`."""
        # The right child is always right next to the left child.
        return 2 * parent + 2

    def _parent(self, child: int) -> int:
        """Return the position of the parent of the node at `child`."""
        # The inverse of the two formulas above; // throws away the remainder,
        # so both children of a node land on the same parent.
        return (child - 1) // 2

    def _swap(self, i: int, j: int) -> None:
        """Exchange the values at positions i and j."""
        # Save one value first; otherwise the first assignment would
        # overwrite it and we would lose it.
        temp = self.storage[i]
        # Copy the value at j into position i.
        self.storage[i] = self.storage[j]
        # Put the saved value into position j.
        self.storage[j] = temp

    def _is_leaf(self, position: int) -> bool:
        """Return True if position has no children in storage. A position has
        a child exactly when its left child is inside the list."""
        # The left child is the first child to appear in the list, so if even
        # that one is past the end, the node has no children at all.
        leaf = self._left_child(position) >= len(self.storage)
        return leaf

    # ---- restoring the heap property

    def _bubble_up(self) -> None:
        """Move the last value in storage up until its parent is not greater
        than it, or it reaches the root."""
        # The value that was just added is the last one in the list.
        position = len(self.storage) - 1
        # Keep climbing while we are not at the root (position 0) and the
        # value is smaller than its parent: that pair breaks the heap property.
        while position > 0 and self.storage[position] < self.storage[self._parent(position)]:
            # Swap the value with its parent, moving it one level up.
            self._swap(position, self._parent(position))
            # Follow the value to its new position and check again.
            position = self._parent(position)

    def _sink_down(self) -> None:
        """Move the value at the root down until neither child is smaller
        than it, or it reaches a leaf."""
        # The value that was just moved to the top sits at position 0.
        position = 0
        # Assume the value still has work to do until we find it does not.
        sinking = True
        # A leaf has no children, so there is nowhere further down to go.
        while sinking and not self._is_leaf(position):
            # Assume the left child is the smaller one. It always exists here,
            # because this position is not a leaf.
            smaller = self._left_child(position)
            # The right child might not exist, so check that it is inside
            # the list before we look at it.
            right = self._right_child(position)
            if right < len(self.storage) and self.storage[right] < self.storage[smaller]:
                # The right child exists and is smaller, so it is the one to compare with.
                smaller = right
            if self.storage[position] > self.storage[smaller]:
                # The value is greater than its smaller child: swap them,
                # moving the value one level down.
                self._swap(position, smaller)
                # Follow the value to its new position and check again.
                position = smaller
            else:
                # Neither child is smaller: the heap property holds, so stop.
                sinking = False

    # ---- the priority queue interface

    def size(self) -> int:
        """Return the number of values in the heap."""
        # The list holds exactly the values in the heap, nothing else.
        count = len(self.storage)
        return count

    def peek(self):
        """Return the smallest value without removing it; None if empty."""
        # Pessimistic default: an empty heap has no smallest value.
        smallest = None
        if len(self.storage) > 0:
            # The heap property puts the smallest value at the root.
            smallest = self.storage[0]
        return smallest

    def add(self, value) -> None:
        """Place value in the heap, keeping the minimum heap property."""
        # Put the new value at the end: the next free spot at the bottom.
        self.storage.append(value)
        # It may be smaller than its parent, so let it float up if needed.
        self._bubble_up()

    def get(self):
        """Remove and return the smallest value; None if empty."""
        # Pessimistic default: an empty heap gives back None.
        smallest = None
        if len(self.storage) > 0:
            # The answer is the root.
            smallest = self.storage[0]
            # Remove the last value from the list so there are no gaps.
            last = self.storage.pop()
            # If that was the only value, the heap is now empty and there is
            # nothing to move; otherwise the last value takes the vacant root.
            if len(self.storage) > 0:
                self.storage[0] = last
                # It is probably too big for the top, so let it sink if needed.
                self._sink_down()
        return smallest

    # ---- printing

    def __len__(self) -> int:
        # Lets len(heap) work: it is the same as heap.size().
        return self.size()

    def __str__(self) -> str:
        # Shows the list, which is the whole tree level by level.
        return f"MinHeap363{self.storage}"

---

## The emergency department

### The model

An emergency department is a priority queue with people in it. Patients are not
seen in the order they arrive; they are seen in the order of how badly they
need to be seen. Every patient has a **severity**. A smaller number is more
severe, which is exactly what a **minimum** heap returns first:

| Severity | Meaning | Who comes out of the heap first |
|---|---|---|
| `1` | life-threatening | always ahead of severities 2 and 3 |
| `2` | severe | ahead of severity 3 |
| `3` | non-life-threatening | last. You may wait 12 hours with a fever. |

Among patients with the **same** severity, the one who **arrived first** is
seen first. (This is not exactly how a real ED works, but it is sufficient for our needs here.)
The next section shows how the heap can know that.

### What goes into the waiting room

The heap holds *patients*, so what, exactly, is a patient? Before choosing, ask
what the simulation needs to know about someone in the waiting room.

- **To maintain patient priority**, the heap needs the severity, and, to break ties among
  equal severities, the minute the patient arrived.
- **When a doctor calls a patient in**, we need the severity (to count them
  under the right severity and to pick the treatment time) and the arrival
  minute (the wait is `minute - arrival_minute`).

That is everything. Think of the wristband a triage nurse puts on a patient: a colour
for how urgent they are, and a time stamped on it. The nurse deciding who is
next needs nothing else, and neither does our simulation.

**Option 1: a `Patient` object.** We could write a class with a severity and an
arrival minute. It would work, but we would also have to teach Python how to
compare two `Patient`s, by overriding `__lt__` (the special method that tells
  Python what `<` means for a class), so that the heap can tell which is more urgent. That is
a lot of code for a thing that has only two facts in it.

**Option 2: a tuple with a unique id**, such as
`(patient_id, severity, arrival_minute)`. Two problems.

- *Order matters inside a tuple.* Python compares tuples element by element, so
  the first element decides. With the id first, the heap would order patients by
  id, which is the order they arrived, and severity would never get a say. The
  waiting room would become a plain first-come, first-served line.
- *The id would never be used.* We do not follow individual patients. We only
  count patients by severity and add up waits. Put the id last and it is carried
  around and never read. It is not even needed to break ties: at most one
  patient arrives per minute, so no two patients share an arrival minute.

**Our choice: a pair, `(severity, arrival_minute)`.** The smallest thing that
holds what we need. The elements are in order of importance, most important
first:

```python
(severity, arrival_minute)
```

- A smaller severity wins, so the life-threatening come out first.
- When two severities tie, the smaller (earlier) `arrival_minute` wins, so
  within a severity the line is first come, first served.
- No extra priority number is needed.

Here is how the pair travels, in code that mirrors what `simulate` does:

```python
room = MinHeap363()
room.add((3, 17))               # a severity-3 patient arrives at minute 17
room.add((1, 20))               # a severity-1 patient arrives at minute 20

severity, arrived = room.get()  # unpack the pair that comes out: (1, 20)
wait = 25 - arrived             # called in at minute 25: waited 5 minutes
```

Taking a pair apart into two names, as in the `severity, arrived = ...` line,
is called *unpacking*. A tuple cannot be changed once it is made, which suits
us: nothing about a patient changes while they wait. (Well, in real life a
patient's condition may deteriorate while they are waiting. Modeling that would
be an interesting expansion of the simulation, but not for this assignment; see
*Possible expansions* at the end.)

`test_heap_tuples`, in the *Testing* section below, checks that the heap
orders pairs of this shape correctly.

**When would you need more?** If we wanted to follow *individual* patients, for
example to report who waited longest, we would need an id, or a `Patient`
object. We don't, so we don't pay for it. Keep in the heap what the heap needs
for ordering plus what you need when the patient comes out, and no more.

### The clock

A discrete event simulation follows a system through the moments when
something happens: a doctor becomes free, a patient arrives. The simplest way
to run one is to advance a clock **one minute at a time**. The simulation is
one master loop, and each minute, in this order:

1. **Treatment.** Every doctor who is free this minute calls in the next
   patient with `get`, if anyone is waiting. That patient waited
   `minute - arrival_minute` minutes; record it under their severity. The
   doctor is now busy for a random number of minutes that depends on the
   severity.
2. **Arrival.** With probability `arrival_prob`, one new patient arrives. Give
   them a random severity. If the waiting room (your `MinHeap363`) already
   holds `WAITING_ROOM_CAPACITY` patients, the patient is **turned away**: they
   are counted, but not added and never seen. Otherwise `add` the pair
   `(severity, minute)`.
3. **Statistics.** After the doctors and the arrival, update the running
   statistics for this minute: the size of the waiting room, how many doctors
   are busy, and whether the room just filled for the first time.

A doctor who starts a treatment at minute `m` that lasts `t` minutes is free
again at minute `m + t`. One list, with one entry per doctor holding *the
minute this doctor is next free*, is all the bookkeeping the doctors need.

Because treatment comes before arrival in each minute, a patient who arrives
in minute `m` can be seen at minute `m + 1` at the earliest. The shortest
possible wait is one minute.

When the clock runs out, some patients may still be in the waiting room.
They were never seen, so they have no wait time to average. **Count them
anyway.** A severity with a short average wait and 400 people still in the
waiting room is not being served well.

### The parameters

The values below are **placeholders** that behave sensibly. They are biased
toward the non-life-threatening and severe patients, as real EDs are. Your code
must read them from these names; never type the numbers into your functions.
Then the simulation keeps working when we replace them.

In [ ]:
# Given -- simulation settings. PLACEHOLDER values.
LIFE_THREATENING = 1
SEVERE = 2
NON_LIFE_THREATENING = 3
TIERS = [LIFE_THREATENING, SEVERE, NON_LIFE_THREATENING]

# Share of arrivals with each severity, in the same order as TIERS. Must add up to 1.
TIER_WEIGHTS = [0.10, 0.35, 0.55]

# Treatment length in minutes for each severity: (shortest, longest), inclusive.
TREATMENT_MINUTES = {
    LIFE_THREATENING: (60, 180),
    SEVERE: (30, 90),
    NON_LIFE_THREATENING: (10, 30),
}

ARRIVAL_PROB = 0.25          # chance of one arrival in any given minute (about 15 per hour)
NUM_DOCTORS = 12
SIM_MINUTES = 7 * 24 * 60    # one week
SEED = 363                   # same seed -> same "random" week, so runs can be compared

# Most patients the waiting room can hold at once. PLACEHOLDER.
WAITING_ROOM_CAPACITY = 60

### Before you build: what is a simulation?

You have probably not written one. Everything you need is here.

**Why simulate.** We want to know how an ED behaves when we change the
staffing or the number of patients. We cannot experiment on a real ED, and
the math for a system with random arrivals, random illnesses and random
treatments is hard. So we build a small imitation, run it on the computer,
and measure it. The imitation does not have to be real. It has to capture
the few things that make waiting happen: *limited doctors, random demand,
and an order of service.*

**State, event, clock.** A simulation has three ingredients:

- **State**: everything that describes the system right now. Ours is the
  waiting room (the heap), the list of when each doctor is free, and the
  counters.
- **Events**: the things that change the state. A patient arrives; a doctor
  starts a treatment; a treatment ends.
- **Clock**: the simulated time, advancing step by step. One simulated week
  runs in a fraction of a second because the clock does not wait.

"Discrete" means the state only changes at separate moments, not
continuously. Ours advances the clock one minute at a time and checks what
happens in that minute. (More elaborate simulators jump straight from one event to
the next. The result is the same; the minute-by-minute version is easier to
get right.)

**The random seed.** A computer does not produce truly random numbers. A
*pseudo-random number generator* computes each number from the previous one,
by a fixed formula, and the first number comes from a starting value called
the **seed**. Two consequences: (1) the numbers look random, but the same
seed always produces exactly the same sequence; (2) a different seed produces
a different sequence. In Python you set the seed with `random.seed(s)`. For
us this is useful. With the same seed, the simulation replays the same week,
so you can find a bug by running the same week again, and you can compare two
staffing levels on *the same* week, so that any difference comes from the
staffing and not from different random numbers.

**One run is one sample.** A run is not the answer; it is one possible week.
A result from one seed may be due to chance. Before you believe a difference, run
several seeds and see whether it holds. Statistics (means, percentiles,
spread) are how you summarize many runs.

**Utilization and the cliff.** If patients bring in, on average, 11 minutes
of work per minute and you have 12 doctors, the doctors are busy 92% of the
time, and the waiting room is usually short, although its length varies from
minute to minute. With 10 doctors, work arrives faster than the doctors can
finish it, and the waiting room grows until it is full, and then patients are turned away. The change between those
two situations is sudden, not gradual (we call it the "cliff"). Queues behave
this way, and it is the main thing this assignment shows you.

#### A hand-trace to do first (10 minutes, on paper)

Before coding, simulate this by hand: **1 doctor**, treatments always exactly
3 minutes, and these arrivals as `(minute, severity)`: `(0, 3)`, `(1, 2)`,
`(2, 3)`, `(3, 1)`, `(4, 3)`. For minutes 0 to 10, write down for every minute
the waiting room (as `(severity, arrival_minute)` pairs), the doctor's
`free_at`, and who is called in, with what wait. Remember the order inside a
minute: treatment first, then arrival. Notice two things: the severity-1
patient who arrived at minute 3 is called ahead of patients who arrived
earlier, and the two severity-3 patients still waiting are called in the order
they arrived. Once your trace and your program agree on a tiny case, you can
trust the program on a large one.

#### How to build it, step by step

1. Read the heap explanation and, when you reach the *Testing* section, run the heap tests.
2. Write the three random helpers; run their tests.
3. Write `simulate` with **one doctor**; print the waiting room each minute
   for the first 20 minutes and compare with your trace.
4. Make the number of doctors a parameter. Then run the tests.
5. Add the waiting-room capacity, then the per-minute statistics, re-running the tests after each.

**Common mistakes**

- Calling `random` inside `simulate`: the draws then change when you reorder code, and results change for no visible reason.
- Using `randint(a, b)` as if `b` were excluded.
- Forgetting patients still in the heap after the loop (they must be counted).
- Forgetting the patients who were turned away: they arrived, so they must still be in the books.
- Updating the statistics in the middle of the minute instead of after both the doctors and the arrival.
- Setting a doctor's `free_at` to the treatment length instead of `minute + length`.
- Looping over doctors but calling `get` on an empty heap: check `size() > 0` each time.
- Comparing means from different seeds, or changing two parameters at once.
- Trusting an average when many patients are still waiting. They have no wait recorded, so they are missing from it.

### Reading: random numbers in Python

Everything random in this simulation comes from Python's `random` module.
Read the
[`random` documentation](https://docs.python.org/3/library/random.html),
at least the sections *Bookkeeping functions*, *Functions for integers* and
*Functions for sequences*. Then try the cell below and make sure you can
explain every line of its output.

Four functions are all you need:

- `random.seed(s)` fixes the starting point of the random number generator.
  The same seed produces the same sequence of "random" numbers every time.
  That is what makes a simulation repeatable, and it is what lets the tests
  check your work.
- `random.random()` returns a float in $[0, 1)$. "With probability $p$" in
  code is `random.random() < p`.
- `random.randint(a, b)` returns an integer from `a` to `b`, **including
  both ends**. (Most of Python excludes the right end; this function does
  not.)
- `random.choices(population, weights=...)` returns a **list** of picks.
  It picks one item by default, so you need `[0]` to get the item itself.

In [ ]:
# Given -- experiment with the random module. Change the seed and re-run.
import random

random.seed(SEED)
print("five floats:     ", [round(random.random(), 3) for _ in range(5)])
print("five dice rolls: ", [random.randint(1, 6) for _ in range(5)])
print("one weighted pick:", random.choices(TIERS, weights=TIER_WEIGHTS))

picks = random.choices(TIERS, weights=TIER_WEIGHTS, k=10_000)
print("share of each tier in 10,000 picks:",
      {tier: picks.count(tier) / len(picks) for tier in TIERS})

### What you write

Three small functions, then the simulation itself. The three small ones are
where all the randomness lives. `simulate` should not call anything from
`random` except `random.seed`.

`simulate` returns its results as a **dictionary**. Python dictionaries look
up a value by name (a *key*) instead of by position. Ours has one entry per
severity, plus one entry, `"room"`, for the waiting room and the doctors. Each
entry is itself a dictionary of counters:

```python
{
    LIFE_THREATENING:     {"arrived": ..., "seen": ..., "total_wait": ...,
                           "still_waiting": ..., "turned_away": ...},
    SEVERE:               {...},
    NON_LIFE_THREATENING: {...},
    "room":               {"peak_waiting": ..., "first_full_minute": ...,
                           "waiting_sum": ..., "busy_sum": ...},
}
```

Here is what a real one looks like. The run is tiny on purpose (one doctor,
forty minutes, a waiting room of three seats) so that the numbers are small
enough to read. Remember that `LIFE_THREATENING` is `1`, `SEVERE` is `2` and
`NON_LIFE_THREATENING` is `3`.

```python
>>> stats = simulate(num_doctors=1, arrival_prob=0.3, minutes=40, capacity=3, seed=1)
>>> stats
{1: {'arrived': 1, 'seen': 1, 'total_wait': 8, 'still_waiting': 0, 'turned_away': 0},
 2: {'arrived': 2, 'seen': 0, 'total_wait': 0, 'still_waiting': 1, 'turned_away': 1},
 3: {'arrived': 7, 'seen': 1, 'total_wait': 1, 'still_waiting': 2, 'turned_away': 4},
 'room': {'peak_waiting': 3, 'first_full_minute': 9, 'waiting_sum': 106, 'busy_sum': 39}}
>>> stats[NON_LIFE_THREATENING]["turned_away"]   # severity, then counter
4
```

Read the severity-3 line: seven non-life-threatening patients arrived. One was
seen, after waiting 1 minute. Two were still in the waiting room when the clock
ran out. Four found the room full and were turned away. And $7 = 1 + 2 + 4$.
Your numbers will be bigger, but they have exactly this shape.

**Why a dictionary, and not a csv file?** A csv file is a table of text stored
on disk. It is the right tool for moving a table between programs (a
spreadsheet, a plotting tool) or for keeping it after your program ends. But
`simulate` does not talk to another program: it hands its answer to the very
next lines of *your* code, which are the report, the tests and the experiment
loop. Sending that answer through a file would mean writing it out, reading it
back, and converting every value from text to a number again, just to ask
"how many severity-3 patients were turned away?". The results are also nested
(a group of counters for each severity, plus one for the room), and a csv is
flat, one row after another. A dictionary keeps the answer in memory, already
numbers, and lets you ask for any value by name: `stats[SEVERE]["seen"]`. If
you later want a graph in a spreadsheet for your report, that is the moment to
write a csv, from the dictionary.

- `arrived`: how many patients of this severity arrived (including those turned away).
- `seen`: how many were called in by a doctor.
- `total_wait`: the sum of the waits of the patients who were seen.
- `still_waiting`: how many were still in the waiting room when the clock ran out.
- `turned_away`: how many were refused because the waiting room was full.

For every severity, `arrived == seen + still_waiting + turned_away`. Nobody
leaves the ED without being counted.

The `"room"` numbers are updated once per minute, after the doctors and the
arrival:

- `peak_waiting`: the largest waiting-room size seen at the end of any minute.
- `first_full_minute`: the first minute at whose end the room held
  `capacity` patients, or `None` if that never happened.
- `waiting_sum`: the sum, over all minutes, of the waiting-room size. Divide by
  the number of minutes for the average queue length.
- `busy_sum`: the sum, over all minutes, of the number of busy doctors. A doctor
  who starts a treatment at minute `m` lasting `t` minutes is busy during
  minutes `m` to `m + t - 1`. Divide by `num_doctors * minutes` for utilization.

Should a life-threatening patient be turned away because the room is full of
severity-3 patients? Our rule says yes: the room is full, whoever arrives.
Question 6 asks what you think of that.

In [ ]:
# TODO -- write the three helpers and simulate().
def random_tier() -> int:
    """Return a random severity from TIERS, chosen according to TIER_WEIGHTS."""
    tier = NON_LIFE_THREATENING
    # TODO
    return tier


def arrival_happens(arrival_prob: float) -> bool:
    """Return True with probability arrival_prob."""
    happens = False
    # TODO
    return happens


def treatment_minutes(tier: int) -> int:
    """Return a random treatment length for tier, within TREATMENT_MINUTES[tier]."""
    minutes = 0
    # TODO
    return minutes


def simulate(num_doctors: int, arrival_prob: float, minutes: int,
             capacity: int = WAITING_ROOM_CAPACITY,
             seed: int | None = None) -> dict:
    """Run the ED for `minutes` minutes and return the statistics.

    Each minute: every free doctor calls in the most urgent waiting patient
    from a MinHeap363; then at most one new patient arrives and is added to
    the heap as (severity, arrival_minute), or turned away if the heap
    already holds `capacity` patients; then the statistics are updated.
    """
    random.seed(seed)
    stats = {tier: {"arrived": 0, "seen": 0, "total_wait": 0,
                    "still_waiting": 0, "turned_away": 0}
             for tier in TIERS}
    stats["room"] = {"peak_waiting": 0, "first_full_minute": None,
                     "waiting_sum": 0, "busy_sum": 0}
    # TODO
    return stats

In [ ]:
# Given -- turning statistics into a report.
TIER_NAMES = {LIFE_THREATENING: "life-threatening",
              SEVERE: "severe",
              NON_LIFE_THREATENING: "non-life-threatening"}


def average_wait(stats: dict, tier: int) -> float | None:
    """Average wait, in minutes, of the patients of `tier` who were seen."""
    average = None
    if stats[tier]["seen"] > 0:
        average = stats[tier]["total_wait"] / stats[tier]["seen"]
    return average


def print_report(stats: dict) -> None:
    print(f"{'severity':22s} {'arrived':>8s} {'seen':>6s} {'waiting':>8s} {'turned away':>12s} {'avg wait (min)':>15s}")
    for tier in TIERS:
        s = stats[tier]
        avg = average_wait(stats, tier)
        shown = "--" if avg is None else f"{avg:.1f}"
        print(f"{TIER_NAMES[tier]:22s} {s['arrived']:8d} {s['seen']:6d} "
              f"{s['still_waiting']:8d} {s['turned_away']:12d} {shown:>15s}")
    room = stats["room"]
    print(f"waiting room: peak {room['peak_waiting']}, first full at minute {room['first_full_minute']}")


print_report(simulate(NUM_DOCTORS, ARRIVAL_PROB, SIM_MINUTES, seed=SEED))

---

## Testing

From here on, the notebook checks your work. The tests below are **Given**:
copy them into your test files unchanged.

> **A note on these tests.** They are here for your convenience, to help you
> find mistakes sooner. The simulation does not need them: nothing in
> `simulate` calls them or depends on them, and you could write a correct
> simulation without ever running one. Use them as a safety net, and as a
> description of what each part should do.

### How a test works

A test is a small function that runs a piece of your code and then states
something that must be true, using `assert`. `assert condition, "message"` does
nothing if the condition is true. If it is false, Python stops that function
with an error carrying the message, and that is how a test says "this is wrong".
`assert` is beyond what we covered in class; you only need to recognise it when
you read a test, you do not need to write one.

Running the tests one at a time would be tedious, so the next cell defines a
helper, `run_tests`. Give it the start of the tests' names, for example
`run_tests("test_heap")`, and it runs every test whose name starts that way,
prints `ok` or `FAIL` for each, and ends with a count. You do not need to read
the helper to use it, but it is commented line by line if you are curious.

In [ ]:
# Given -- the test runner. Use it as run_tests("test_heap") or run_tests("test_ed").
def run_tests(prefix: str) -> None:
    """Run every test function whose name starts with `prefix`, print `ok` or
    `FAIL` for each, and finish with a count of how many passed."""
    # globals() is a dictionary of everything defined so far in this notebook:
    # every function and variable, stored under its name. .items() hands us each
    # (name, thing) pair, and sorted() puts them in alphabetical order so the
    # tests always run in the same order.
    # We keep a pair only if the name starts with `prefix` and the thing is a
    # function (callable), so that is how we find the tests without listing them.
    tests = [(name, fn) for name, fn in sorted(globals().items())
             if name.startswith(prefix) and callable(fn)]
    # Count the tests that pass, starting from zero.
    passed = 0
    for name, fn in tests:
        # try / except: run the test, but if it fails do not crash the whole
        # notebook. A failing assert makes Python "raise" an error. The code in
        # the try block is what we attempt; the code in except runs only if it fails.
        try:
            # Calling the test function. If every assert in it holds, it just
            # returns and prints nothing.
            fn()
            # We only reach this line when the test did not raise an error.
            print(f"ok    {name}")
            passed += 1
        except Exception as err:   # AssertionError, TypeError, IndexError...
            # `err` is the error that was raised. type(err).__name__ is its kind
            # (for a failed assert, AssertionError) and str(err) is the message
            # written after the comma in the assert, which says what went wrong.
            print(f"FAIL  {name}: {type(err).__name__}: {err}")
    # The summary line. The \n at the start leaves a blank line before it.
    print(f"\n{passed} of {len(tests)} checks passed")

### Tests for the heap

You do not need to change the heap, and the simulation does not need these
tests. They are here for your convenience: run them if you want to trust the
heap, and read them, because each one is a small, exact description of how the
heap behaves.

**What the tests check:**

| Group | Tests | The promise |
|---|---|---|
| The basics | `empty`, `single_value` | An empty heap fails safely (`None`, nothing changes); a heap with one value gives it back. |
| The two repairs | `bubble_up_class_example`, `sink_down_swaps_with_smaller_child`, `is_leaf` | The helpers of `add` and `get` do their job in small cases you can check by hand. |
| Order | `ascending_and_descending_adds`, `duplicates`, `random_matches_sorted`, `tuples` | Whatever order values go in, they come out smallest first, including repeated values and pairs. |
| The rule itself | `property_holds` | After every operation, every parent is less than or equal to both children. |

**Commands in these tests that we did not cover in class** (`assert` is
explained at the top of this *Testing* section):

- `random.seed(n)`: starts the random numbers from a known point, so that a
  test that uses randomness gives the same result every time it is run.
- `f"...{x}..."` (an *f-string*): puts the value of `x` into the text. The
  tests use it so that a failure message shows you the actual numbers.

- `h.storage = [...]`: a test may reach inside the heap and set its list
  directly, to build an exact situation without calling `add` many times.
  Code that *uses* a heap should never do this; a test may, because it is
  checking the inside.
- `h._is_leaf(0)`: a name that starts with an underscore marks a helper meant
  for the class itself. A test calls it directly only to check it alone.
- `sorted(values)`: a new list with the same values in order. The tests use it
  as the known right answer to compare the heap against.
- `x is None` / `x is not None`: asks whether `x` is exactly `None`.
- `for _ in range(n)` / `for _ in values`: repeat once per item; the underscore
  means we do not need the loop variable.

In [ ]:
# Given -- tests for the heap.
# The imports bring in two modules (ready-made sets of tools): `random` for
# random numbers and `inspect`, which a later test uses to read the source code
# of a function.
import inspect
import random


def heap_property_ok(heap: MinHeap363) -> bool:
    """True if every parent in heap.storage is <= both of its children."""
    # Not a test: a checking tool that several tests share. It looks directly
    # at the list, so it does not trust the heap's own methods to judge them.
    a = heap.storage
    # Assume the property holds until we find a pair that breaks it.
    ok = True
    # Visit every position except the root. Each one is a child of exactly one
    # parent, so checking each child against its parent checks every pair.
    for child in range(1, len(a)):
        # (child - 1) // 2 is the parent position. A parent greater than its
        # child breaks the minimum heap property.
        if a[(child - 1) // 2] > a[child]:
            ok = False
    return ok


# ---- The basics


def test_heap_empty():
    # Why: the failure cases in the interface table must behave as promised.
    h = MinHeap363()
    assert h.size() == 0, "a new heap has size 0"
    # `is None` asks whether the answer is exactly None, "no value".
    assert h.peek() is None, "peek on an empty heap returns None"
    assert h.get() is None, "get on an empty heap returns None"
    # A failed operation must leave everything as it was.
    assert h.size() == 0, "a failed get changes nothing"


def test_heap_single_value():
    # Why: the smallest possible heap is where an off-by-one hides, because the
    # only value is both the root and the last value in the list.
    h = MinHeap363()
    h.add(42)
    assert h.size() == 1
    assert h.peek() == 42, "peek returns the only value"
    # peek only looks; if it removed the value, the size would drop to 0.
    assert h.size() == 1, "peek does not remove"
    assert h.get() == 42, "get returns the only value"
    assert h.size() == 0, "and the heap is empty again"
    # Getting from the now-empty heap is the failure case again.
    assert h.get() is None


# ---- The two repairs


def test_heap_is_leaf():
    # Why: _is_leaf decides when _sink_down stops. We test it alone, on a list
    # we build by hand: [1, 2, 3, 4] is the pyramid
    #         1
    #       2   3
    #     4
    h = MinHeap363()
    # Setting the list directly, without add, builds this exact situation.
    h.storage = [1, 2, 3, 4]
    assert not h._is_leaf(0), "the root has two children"
    # Position 1 has its left child at 2 * 1 + 1 = 3, which is in the list.
    assert not h._is_leaf(1), "position 1 has a left child at 3"
    # Position 2 would have children at 5 and 6, past the end of the list.
    assert h._is_leaf(2), "position 2 would have children at 5 and 6"
    assert h._is_leaf(3), "position 3 is the last leaf"


def test_heap_bubble_up_class_example():
    # From class: 2, 8, 4 is a minimum heap; adding 1 floats to the root.
    # Why: this follows one value up two levels. The expected list can be
    # checked by hand: 1 lands at position 3, swaps with its parent 8 (position
    # 1), then with its parent 2 (position 0).
    h = MinHeap363()
    for v in [2, 8, 4]:
        h.add(v)
    h.add(1)
    # Comparing whole lists checks every position, not just the root.
    assert h.storage == [1, 2, 4, 8], f"expected [1, 2, 4, 8], got {h.storage}"


def test_heap_sink_down_swaps_with_smaller_child():
    # 3 moves to the root and must swap with 1 (the smaller child), not 2.
    # Why: swapping with the larger child would put 2 above 1 and break the
    # rule. Only a heap that picks the smaller child passes.
    h = MinHeap363()
    # A valid minimum heap built by hand:   0
    #                                     2   1
    #                                   3
    h.storage = [0, 2, 1, 3]
    # get removes 0, moves the last value (3) to the root, and sinks it.
    assert h.get() == 0
    assert h.storage == [1, 2, 3], f"expected [1, 2, 3], got {h.storage}"


# ---- Order


def test_heap_ascending_and_descending_adds():
    # Why: already-sorted input is the easiest case for add (nothing moves) and
    # reverse-sorted input is the hardest (everything floats to the top).
    # Both must give the same ordered output.
    for values in ([1, 2, 3, 4, 5, 6, 7], [7, 6, 5, 4, 3, 2, 1]):
        h = MinHeap363()
        for v in values:
            h.add(v)
        # One get per value added; `for _ in values` repeats len(values) times.
        out = [h.get() for _ in values]
        assert out == [1, 2, 3, 4, 5, 6, 7], f"added {values}, got {out}"


def test_heap_duplicates():
    # Why: the comparisons are strict (< and >), so equal values must not swap
    # forever or get lost. Repeated values must all come back.
    h = MinHeap363()
    for v in [2, 1, 2, 1, 2]:
        h.add(v)
    out = [h.get() for _ in range(5)]
    assert out == [1, 1, 2, 2, 2], f"got {out}"


def test_heap_random_matches_sorted():
    # Why: hand-picked examples can miss a case. Here we compare the heap with
    # a method we trust, Python's sorted(), on many random lists. The sizes 31,
    # 32 and 33 sit around a full bottom row of the pyramid (31 = 1+2+4+8+16
    # values), where mistakes with the last value tend to show up.
    random.seed(9)
    for n in [2, 3, 10, 31, 32, 33, 200]:
        # n random values between 0 and 100; with n = 200 there are many repeats.
        values = [random.randint(0, 100) for _ in range(n)]
        h = MinHeap363()
        for v in values:
            h.add(v)
        assert h.size() == n
        out = [h.get() for _ in range(n)]
        # sorted(values) is the known right answer: the same values in order.
        assert out == sorted(values), f"n={n}: values came out out of order"
        # Every value has been taken out, so one more get fails safely.
        assert h.get() is None


def test_heap_tuples():
    # The simulation stores (severity, arrival_minute) pairs. Python compares tuples
    # element by element, so a smaller first element wins and ties go to the second.
    # Why: this is the exact use the ED makes of the heap. Within one severity,
    # the patient who arrived earlier must come out first.
    h = MinHeap363()
    for patient in [(3, 1), (2, 2), (1, 3), (2, 4), (1, 5)]:
        h.add(patient)
    out = [h.get() for _ in range(5)]
    # Severity 1 first, oldest first: (1, 3) before (1, 5); then the 2s; then 3.
    assert out == [(1, 3), (1, 5), (2, 2), (2, 4), (3, 1)], f"got {out}"


# ---- The rule itself


def test_heap_property_holds():
    # Why: the other tests look at what comes out. This one looks at the list
    # itself after every single operation, so a heap that is broken inside,
    # even if it happens to give right answers, is caught.
    random.seed(363)
    h = MinHeap363()
    # 300 random operations, a mix of adds and gets.
    for _ in range(300):
        # random.random() is between 0 and 1: add about 60% of the time, and
        # get the other 40%, so the heap grows but also shrinks.
        if random.random() < 0.6:
            h.add(random.randint(-50, 50))
        else:
            h.get()
        # The checking tool above must still say yes, whatever just happened.
        assert heap_property_ok(h), f"heap property broken: {h.storage}"


# Run every function above whose name starts with "test_heap".
run_tests("test_heap")

### Tests for the simulation

*These tests are a convenience for you. The simulation itself does not need
them; they are a safety net.*

**The difficulty.** Your simulation is random, so no test can say "the answer
must be exactly 4.7 minutes". Instead, each test below asserts something that
must be true *whatever the dice say*, or something that is true *on average*
and allows a small margin. When a test fails, the message tells you which
promise was broken.

**What the tests check**, in the order they appear:

| Group | Tests | The promise |
|---|---|---|
| The random helpers | `random_tier`, `arrival_happens`, `treatment_minutes` | Each helper produces what its docstring says: the right shares, the right rate, the right range. |
| The bookkeeping | `nobody_arrives`, `everyone_is_counted` | Nobody is lost: every arrival is seen, still waiting, or turned away. |
| The waiting room | `small_waiting_room...`, `big_waiting_room...` | The room never holds more than its capacity, and turns patients away only when it is full. |
| The clock and the doctors | `plenty_of_doctors...`, `statistics_are_possible`, `overloaded_doctor...` | The statistics stay within what is physically possible. |
| The priority queue | `priority_matters_when_busy`, `uses_the_heap` | Urgent patients are seen first, and the waiting room really is the heap. |
| The seed | `same_seed_same_week` | The same seed reproduces the same week exactly. |

**Read the tests as well as run them.** Each one is a small, exact description
of a behaviour you are about to program, and reading it before you write that
part tells you what is expected. When one fails, read its message, then read
the test, then look at your code.

**Commands in the tests that we did not cover in class.** You never have to
write these; you only have to recognise them.

- `assert condition, "message"`: does nothing if the condition is true; if it
  is false, the test fails and shows the message.
- `random.seed(n)`: starts the random numbers from a known point, so that a
  test that uses randomness gives the same result every time it is run. Without
  it, a test could pass today and fail tomorrow.
- `any(...)` and `all(...)`: `any` is true if at least one item in a group is
  true; `all` is true only if every item is true.
- `zip(a, b)`: walks two lists side by side, pairing the first with the first,
  the second with the second, and so on.
- `abs(x)`: the size of `x` ignoring its sign. `abs(share - weight) < 0.02`
  means "share and weight differ by less than 0.02, in either direction".
- `{... for ...}` (a *set comprehension*) and `[... for ...]` (a *list
  comprehension*): a compact way to build a set or a list by repeating an
  expression. `f"...{x}..."` is an f-string: it puts the value of `x` into the
  text, which is how failure messages show you the numbers.
- `inspect.getsource(simulate)`: gives the program text of a function as a
  string, so a test can check what the function is made of.

In [ ]:
# Given -- tests for the simulation. Run this cell after every change.
#
# How to read a test: the lines before the asserts set up a situation (a
# number of doctors, an arrival rate, a number of minutes, a seed); the asserts
# say what must be true afterwards. The seed is different in each test only so
# that the tests do not all examine the same week.


# ---- The random helpers
# Each helper is tested on its own, before simulate() uses it, so a mistake in a
# helper is found here and not hidden inside a whole simulated week.


def test_ed_random_tier_follows_weights():
    # Why: random_tier must pick severity 1, 2 and 3 in the proportions of
    # TIER_WEIGHTS. A single pick proves nothing, so we make many.
    # Fixing the seed makes this test give the same answer every time.
    random.seed(1)
    # 20,000 picks. `for _ in range(...)` repeats the call that many times; the
    # underscore means we do not need the loop counter.
    picks = [random_tier() for _ in range(20_000)]
    # zip pairs each severity with its weight: (1, 0.10), (2, 0.35), (3, 0.55).
    for tier, weight in zip(TIERS, TIER_WEIGHTS):
        # The share of picks that were this severity.
        share = picks.count(tier) / len(picks)
        # Random results never match a weight exactly, so we accept a margin of
        # 0.02. With 20,000 picks, a correct helper stays well inside it; a
        # helper that ignores the weights does not.
        assert abs(share - weight) < 0.02, f"tier {tier}: share {share:.3f}, expected about {weight}"


def test_ed_arrival_happens_extremes_and_rate():
    # Why: arrival_happens(p) must be True with probability p. The two extremes
    # have exact answers, so we test those exactly; the middle we test as a rate.
    random.seed(2)
    # Probability 0 must never happen. any(...) is true if even one call came
    # back True, so `not any(...)` says none of the 1000 calls did.
    assert not any(arrival_happens(0.0) for _ in range(1000)), "probability 0 never happens"
    # Probability 1 must always happen: all(...) needs every one of the 1000.
    assert all(arrival_happens(1.0) for _ in range(1000)), "probability 1 always happens"
    # In between, count the Trues (True counts as 1 when we add) and divide.
    share = sum(arrival_happens(0.25) for _ in range(20_000)) / 20_000
    # About a quarter of the calls should be True, give or take 0.02.
    assert abs(share - 0.25) < 0.02, f"probability 0.25 happened {share:.3f} of the time"


def test_ed_treatment_minutes_in_range():
    # Why: a treatment time must fall inside the interval for its severity, and
    # the ends of the interval must be reachable (an off-by-one in randint or
    # in your range would make the longest or shortest time impossible).
    random.seed(3)
    for tier in TIERS:
        # The interval for this severity, from the settings cell.
        low, high = TREATMENT_MINUTES[tier]
        # Draw 5000 times; the braces make a set, which keeps each different
        # value once, so `seen` is the collection of all times that occurred.
        seen = {treatment_minutes(tier) for _ in range(5000)}
        # Nothing shorter than low or longer than high ever appeared.
        assert min(seen) >= low and max(seen) <= high, f"tier {tier}: out of [{low}, {high}]"
        # And both ends did appear: 5000 draws from an interval of at most 121
        # values will almost certainly hit both.
        assert low in seen and high in seen, f"tier {tier}: both ends must be possible"


# ---- The bookkeeping


def test_ed_nobody_arrives():
    # Why: with arrival_prob = 0 the ED stays empty, so every counter must be
    # exactly zero. This catches counters that start at the wrong value, or
    # that change without a patient.
    stats = simulate(num_doctors=3, arrival_prob=0.0, minutes=600, seed=4)
    for tier in TIERS:
        # Comparing two dictionaries: equal only if every key has the same value.
        assert stats[tier] == {"arrived": 0, "seen": 0, "total_wait": 0, "still_waiting": 0, "turned_away": 0}
    room = stats["room"]
    # An empty room has no peak and never became full; `is None` is how we ask
    # whether first_full_minute still holds None ("it never happened").
    assert room["peak_waiting"] == 0 and room["first_full_minute"] is None
    assert room["waiting_sum"] == 0 and room["busy_sum"] == 0


def test_ed_everyone_is_counted():
    # Why: this is the most important test. Nobody may vanish: every patient who
    # arrived is in exactly one of three groups when the clock stops.
    # 4 doctors and a busy arrival rate mean all three groups will occur.
    stats = simulate(num_doctors=4, arrival_prob=0.3, minutes=2000, seed=5)
    # About 0.3 x 2000 = 600 patients should arrive. More than 400 shows that
    # arrivals really happen at the rate you were given.
    assert sum(stats[t]["arrived"] for t in TIERS) > 400, "about 600 arrivals expected"
    for tier in TIERS:
        s = stats[tier]
        # arrived = seen + still waiting + turned away. If this fails, a patient
        # was counted twice or not at all: check the turned-away branch, and
        # that you count the patients still in the heap after the loop.
        assert s["arrived"] == s["seen"] + s["still_waiting"] + s["turned_away"], f"tier {tier}: {s}"


# ---- The clock and the doctors


def test_ed_plenty_of_doctors_means_one_minute_wait():
    # Why: when there is no shortage of doctors, the wait comes only from the
    # order of the steps inside one minute. Treatment comes before arrival in
    # each minute, so a patient who arrives in minute m is called in at m + 1.
    # (1000 doctors, but at most one arrival per minute.)
    stats = simulate(num_doctors=1000, arrival_prob=0.5, minutes=500, seed=6)
    for tier in TIERS:
        s = stats[tier]
        # Make sure the test is meaningful: this severity did have patients.
        assert s["arrived"] > 0
        # Everyone waited exactly 1 minute, so the total wait equals the number
        # of patients seen (1 minute each).
        assert s["total_wait"] == s["seen"], f"tier {tier}: everyone should wait exactly 1 minute: {s}"
    # Only a patient who arrived in the very last minute can still be waiting,
    # because the clock stopped before the doctors could call them in.
    assert sum(stats[t]["still_waiting"] for t in TIERS) <= 1, "only the last minute's arrival can still be waiting"


def test_ed_statistics_are_possible():
    # Why: the per-minute statistics have hard limits. Here they are checked
    # against what could physically happen, which catches updating them at the
    # wrong moment, or twice a minute.
    doctors, minutes = 4, 2000
    stats = simulate(num_doctors=doctors, arrival_prob=0.3, minutes=minutes, capacity=20, seed=5)
    room = stats["room"]
    # Doctor-minutes busy: more than none (they had patients), and never more
    # than every doctor being busy in every minute. A chained comparison
    # `0 < x <= y` means both `0 < x` and `x <= y`.
    assert 0 < room["busy_sum"] <= doctors * minutes, f"busy_sum {room['busy_sum']}"
    # Waiting-room size added up over the minutes: never more than a full room
    # (20 patients) in every minute.
    assert 0 < room["waiting_sum"] <= 20 * minutes, f"waiting_sum {room['waiting_sum']}"
    # The peak is at least 1 (somebody waited) and never above the capacity.
    assert 0 < room["peak_waiting"] <= 20


def test_ed_overloaded_doctor_is_always_busy():
    # Why: one doctor facing a flood of patients should never be idle, so
    # busy_sum should be nearly one for every minute. It checks that you count a
    # doctor as busy for the whole treatment, not just the first minute of it.
    stats = simulate(num_doctors=1, arrival_prob=0.9, minutes=3000, capacity=10, seed=3)
    # 95% of 3000 minutes, not 100%: the doctor really is idle in the first
    # minute, before the first patient has arrived.
    assert stats["room"]["busy_sum"] >= 0.95 * 3000, f"busy_sum {stats['room']['busy_sum']} of 3000"


# ---- The waiting room


def test_ed_small_waiting_room_turns_patients_away():
    # Why: one doctor, half a patient per minute and a room of 5 seats is far
    # more than the ED can handle, so the room must fill and refuse people.
    stats = simulate(num_doctors=1, arrival_prob=0.5, minutes=1000, capacity=5, seed=9)
    room = stats["room"]
    # Somebody must have been turned away.
    assert sum(stats[t]["turned_away"] for t in TIERS) > 0, "a 5-seat room with 1 doctor must turn patients away"
    # The peak must be exactly the capacity: reaching it shows the room filled,
    # and never exceeding it shows nobody was added to a full room.
    assert room["peak_waiting"] == 5, f"peak {room['peak_waiting']}, expected exactly the capacity, 5"
    # The room filled at some minute, so first_full_minute is no longer None.
    assert room["first_full_minute"] is not None
    # Turned-away patients still arrived, so they still have to balance.
    for tier in TIERS:
        s = stats[tier]
        assert s["arrived"] == s["seen"] + s["still_waiting"] + s["turned_away"], f"tier {tier}: {s}"


def test_ed_big_waiting_room_never_fills():
    # Why: the opposite case. With 1000 doctors the room is almost always
    # empty, so nobody may be turned away, even though the capacity is only 50.
    # It catches a turn-away rule that fires when it should not.
    stats = simulate(num_doctors=1000, arrival_prob=0.5, minutes=500, capacity=50, seed=6)
    # all(...) is true when every severity has zero turned away.
    assert all(stats[t]["turned_away"] == 0 for t in TIERS)
    # The room was never full, so first_full_minute is still None.
    assert stats["room"]["first_full_minute"] is None
    # A patient waits one minute at most, and only one arrives per minute.
    assert stats["room"]["peak_waiting"] <= 1, "with 1000 doctors at most one patient waits, for one minute"


# ---- The priority queue


def test_ed_priority_matters_when_busy():
    # Why: the whole point of the heap. In a busy week, severe patients must be
    # seen sooner than mild ones. 11 doctors is busy but not overwhelmed, so
    # every severity gets seen and has an average wait to compare.
    stats = simulate(num_doctors=11, arrival_prob=0.25, minutes=SIM_MINUTES, seed=7)
    # The average wait of each severity, in the order of TIERS.
    waits = [average_wait(stats, tier) for tier in TIERS]
    # average_wait gives None when nobody of a severity was seen.
    assert None not in waits, f"every tier should have patients seen: {waits}"
    # A life-threatening patient should not wait more than half an hour.
    assert waits[0] < 30, f"life-threatening waited {waits[0]:.0f} min on average"
    # The chain says: severity 1 waits less than 2, and 2 waits less than 3.
    # If the waits come out the other way round or all equal, the heap is not
    # being asked for the most urgent patient.
    assert waits[0] < waits[1] < waits[2], f"waits should grow with the tier number: {waits}"


def test_ed_uses_the_heap():
    # Why: the assignment is about using the heap, and a sorted list would
    # produce the same numbers, so we look at the source code of simulate itself.
    # inspect.getsource returns the text of the function as one long string.
    src = inspect.getsource(simulate)
    # The waiting room must be created from our class.
    assert "MinHeap363()" in src, "the waiting room must be a MinHeap363"
    # And the shortcuts must be absent from the text.
    for banned in ["heapq", "sorted(", ".sort("]:
        assert banned not in src, f"simulate may not use {banned}"


# ---- The seed


def test_ed_same_seed_same_week():
    # Why: simulate calls random.seed(seed), so two runs with the same seed
    # must produce exactly the same week. This is what makes experiments
    # repeatable. It fails when randomness comes from somewhere simulate did
    # not seed, or when the results depend on anything that changes between runs.
    a = simulate(num_doctors=10, arrival_prob=0.25, minutes=3000, seed=8)
    b = simulate(num_doctors=10, arrival_prob=0.25, minutes=3000, seed=8)
    # The two results are dictionaries, so == compares every value inside.
    assert a == b, "the same seed must reproduce the same results"


# Run every function above whose name starts with "test_ed".
run_tests("test_ed")

### Experiments

Once the simulation tests pass, run the cell below. It keeps the week, the seed
and the arrival rate fixed, and changes only the number of doctors.

In [ ]:
# Given -- how many doctors does this ED need?
print(f"arrival_prob = {ARRIVAL_PROB}, one week, seed = {SEED}\n")
print(f"{'doctors':>7s}  " + "  ".join(f"{TIER_NAMES[t]:>22s}" for t in TIERS))
for doctors in range(6, 17):
    stats = simulate(doctors, ARRIVAL_PROB, SIM_MINUTES, seed=SEED)
    cells = []
    for tier in TIERS:
        avg = average_wait(stats, tier)
        shown = "--" if avg is None else f"{avg:.0f}"
        cells.append(f"{shown:>10s} min, {stats[tier]['still_waiting']:4d} left, {stats[tier]['turned_away']:4d} out")
    print(f"{doctors:7d}  " + "  ".join(f"{c:>22s}" for c in cells))

Now add your own experiment cells below this one. Change one thing at a
time and keep the seed fixed, so that the only difference between two runs
is the thing you changed. Then answer these questions in your report, in
your own words:

1. **The cliff.** In the table above, the waits do not grow smoothly as
   doctors are removed. Somewhere they jump. Where is the jump? Using
   `ARRIVAL_PROB`, `TIER_WEIGHTS` and `TREATMENT_MINUTES`, estimate how many
   minutes of treatment arrive per minute of clock time on average. How does
   that number explain where the jump is?
2. **Who pays for a short staff?** When there are too few doctors, which
   severity's wait grows first and fastest? Which barely notices? Why does a
   priority queue produce exactly this pattern? Then compare, in
   the row for the fewest doctors, the average wait of the patients who were
   seen with the number turned away and still waiting. What does the average
   leave out?
3. **Half the staff calls in sick.** Start from a staffing level that keeps
   every severity's average wait under an hour, then halve it. Describe what
   happens to each severity.
4. **A busier night.** Keep `NUM_DOCTORS` fixed and raise `ARRIVAL_PROB`.
   How much busier can the ED get before the non-life-threatening patients
   are effectively never seen within the week?
5. **What the heap does.** Over a simulated week, about how many times is
   `add` called, and how many times is `get` called? If the waiting room
   were an unsorted list scanned for its most urgent patient instead, what
   would each `get` cost, and why does it matter more on a bad day than on a
   quiet one?
6. **Turning patients away.** Our rule turns away whoever arrives when the room is
   full, whatever their severity. Is that acceptable? Look at the `turned_away`
   counts by severity in a bad week. What would you change, and what would it cost?

### Possible expansions (optional)

None of these is required. Each one grows the simulation in a new direction.

- **Not every hour is the same.** Make `ARRIVAL_PROB` change with the hour:
  busier in the evening, quieter at 4 a.m.
- **Make room for the worst cases.** Let a life-threatening patient who arrives at
  a full room take the place of the least urgent patient waiting.
- **Patients who get worse.** In real life a patient's condition can deteriorate
  while they wait. Let a waiting patient's severity increase over time. A
  tuple in the heap cannot be edited, so what would you put in the heap
  instead, and how would the heap learn that a patient's priority changed?
- **Follow individual patients.** Give each patient a unique id (or make a
  `Patient` object) and report who waited longest. What would you have to
  change in what goes into the heap?
- **Surviving the wait.** Give each severity a maximum safe wait. Count the
  patients who are seen after it, or never seen.

## What to submit

Turn in **Python files and one PDF**. Not a notebook.

- **`heap363.py`**: the given `MinHeap363` class, unchanged.
- **`ed_simulation.py`**: the settings, the three random helpers, `simulate`,
  and `average_wait` / `print_report`. It must `from heap363 import MinHeap363`,
  and it must run from a terminal with `python ed_simulation.py` and print the
  report without errors.
- **`test_heap363.py`** and **`test_ed_simulation.py`**: the Given tests,
  unchanged. They must pass; the output of running them is part of your grade.
- **`report.pdf`**: your answers to the six questions, with any table or graph
  that supports a claim and a caption saying what the reader should see.
  Explain the seed in your own words: what it is, why the same seed reproduces
  the same week, and what a result from a single seed can and cannot tell you.
  One to two pages.